In [2]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
rc('axes', fc='w')
rc('figure', fc='w')
rc('savefig', fc='w')
rc('axes', axisbelow=True)
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

In [3]:
catalog_res = requests.get('https://catalog.northeastern.edu/course-descriptions/phth/')
catalog_html = catalog_res.text

soup = BeautifulSoup(catalog_html)

In [4]:
COURSE_CODE_RE = re.compile(r'\b[A-Z&]{2,}\s?\d{4}\b')

def norm(s: str | None) -> str | None:
    """Normalize whitespace and non-breaking spaces."""
    if s is None:
        return None
    return " ".join(s.replace("\xa0", " ").split())

def extract_prereq_codes(prereq_p):
    """
    Given the <p> tag that contains prerequisites, return a list of course codes.
    Works for both <a> links and plain text course codes.
    """
    if prereq_p is None:
        return []

    found = []
    seen = set()

    # 1) linked prereqs
    for a in prereq_p.find_all('a'):
        code = norm(a.get_text(strip=True))
        if code and COURSE_CODE_RE.fullmatch(code) and code not in seen:
            seen.add(code)
            found.append(code)

    # 2) unlinked prereqs from the raw text
    text = norm(prereq_p.get_text(" "))
    if text:
        for code in COURSE_CODE_RE.findall(text):
            code = norm(code)
            if code not in seen:
                seen.add(code)
                found.append(code)

    return found

def parse_courses(soup):
    """
    Parse all course blocks from the HTML soup and return a list of dicts.
    Each dict includes: course_crn, course_title, credits, description, prerequisites (list of strings).
    """
    course_info = []

    for class_i in soup.find_all('div', {'class': 'courseblock'}):
        # course header
        title_p = class_i.find('p', {'class': 'courseblocktitle noindent'})
        if not title_p:
            continue
        class_title_total_i = norm(title_p.get_text())
        try:
            crn_i, title_i, credit_i = class_title_total_i.split('. ', 2)
        except ValueError:
            # sometimes format may differ
            continue
        # credits
        credit_i = credit_i.strip()
        credit_i = re.sub(r"[()]", '', credit_i)
        
        # description
        desc_p = class_i.find('p', class_='cb_desc')
        desc_i = norm(desc_p.get_text()) if desc_p else None

        # find prereq <p>
        prereq_p = None
        for p in class_i.find_all('p', class_='courseblockextra noindent'):
            strong = p.find('strong')
            head = norm(strong.get_text()) if strong else ''
            if head and head.lower().startswith('prerequisite'):
                prereq_p = p
                break

        prereqs = extract_prereq_codes(prereq_p)

        course_info.append({
            "course_crn": crn_i,
            "course_title": title_i,
            "credits": credit_i,
            "description": desc_i,
            "prerequisite": prereqs
        })

    return pd.DataFrame(course_info)


In [5]:
catalog_html = catalog_res.text
soup = BeautifulSoup(catalog_html)

courses = parse_courses(soup)
print(courses[["course_crn", "credits", "prerequisite"]])  

   course_crn  credits prerequisite
0   PHTH 1260  4 Hours           []
1   PHTH 1261  4 Hours           []
2   PHTH 1270  4 Hours           []
3   PHTH 2210  4 Hours           []
4   PHTH 2300  4 Hours           []
..        ...      ...          ...
67  PHTH 8986  0 Hours           []
68  PHTH 9000  0 Hours           []
69  PHTH 9990  0 Hours  [PHTH 9000]
70  PHTH 9991  0 Hours  [PHTH 9990]
71  PHTH 9996  0 Hours  [PHTH 9991]

[72 rows x 3 columns]


### P4 (b)

In [6]:
from pyvis.network import Network
G = nx.DiGraph()

In [7]:
def curriculum_graph(df):
    for row in courses.itertuples():
        G.add_node(
            row.course_crn,
            title = row.course_title,
            credits = row.credits,
            desc = row.description
        )
    
    for row in courses.itertuples():
        for prereq in row.prerequisite:
            if prereq not in G:
                G.add_node(prereq)
            G.add_edge(prereq, row.course_crn)
    
    return G

def draw_curriculum_graph_pyvis(G, output="curriculum_graph.html"):
    net = Network(height="750px", width="100%", directed=True, notebook=False)

    for n, d in G.nodes(data=True):
        net.add_node(n, label=n, title=d.get("title",""), color="lightblue", size=12)
    for u, v in G.edges():
        net.add_edge(u, v, color="gray", arrowStrikethrough=False)

    # simple physics
    net.force_atlas_2based()

    # Avoid the notebook templating path entirely:
    net.write_html(output)   # <- use this instead of net.show()
    print(f"Graph saved to {output}")


In [8]:
G = curriculum_graph(courses)

draw_curriculum_graph_pyvis(G, "curriculum_graph.html")

Graph saved to curriculum_graph.html


### P4(c)

In [9]:
from urllib.parse import urljoin
departments_req = requests.get('https://catalog.northeastern.edu/course-descriptions/')
departments_html = catalog_res.text
dept_soup = BeautifulSoup(departments_html)

In [10]:
BASE = "https://catalog.northeastern.edu"
departments = []
def get_dept(dept_soup):
    uls = dept_soup.find_all('ul', {'class': 'nav levelone'})
    for li in uls[0].find_all('li'):
        a = li.find('a', href=True)
        dept_href = a['href']
        dept_raw = a.get_text().strip()
        dept_name = dept_raw.split('(')[0].replace(' -\u200b', '').strip()
        dept_abbr = dept_href.strip('/').split('/')[-1].upper()
    
        
        dept_url = urljoin(BASE, dept_href)
        departments.append({
            'dept_name': dept_name,
            'dept_abbr': dept_abbr,
            'dept_url': dept_url
        })
    return pd.DataFrame(departments)

In [11]:
dept_df = get_dept(dept_soup)

print(dept_df)

                                  dept_name dept_abbr  \
0                                Accounting      ACCT   
1                            Accounting CPS       ACC   
2        Advanced Manufacturing Systems CPS       AVM   
3                           African Studies      AFRS   
4                          Africana Studies      AFCS   
..                                      ...       ...   
226            Technical Communications CPS       TCC   
227               Telecommunication Systems      TELE   
228                   Technology Leadership      TELR   
229                                 Theatre      THTR   
230  Women’s, Gender, and Sexuality Studies      WMNS   

                                              dept_url  
0    https://catalog.northeastern.edu/course-descri...  
1    https://catalog.northeastern.edu/course-descri...  
2    https://catalog.northeastern.edu/course-descri...  
3    https://catalog.northeastern.edu/course-descri...  
4    https://catalog.northeast

In [12]:
all_dept = []
def get_all_courses(dept_df):
    for i, row in dept_df.iterrows():
        name = row['dept_name']
        abbr = row['dept_abbr']
        url = row['dept_url']
    
        try:
            dept_req_i = requests.get(url)
            dept_html_i = dept_req_i.text
            dept_soup_i = BeautifulSoup(dept_html_i)
            df = parse_courses(dept_soup_i)
            df.insert(0, 'subject', abbr)
            df.insert(1, 'department name', name)

            all_dept.append(df)
            
        except Exception as e:
                print(f"!! Failed on {abbr} ({url}): {e}")
    
    courses = pd.concat(all_dept).drop_duplicates(subset=['course_crn', 'course_title'])
    return courses

In [13]:
courses_df = get_all_courses(dept_df)
courses_df.head()

# save courses
courses_df.to_parquet('neu_courses.parquet', index=False)

In [17]:
courses_df = pd.read_parquet('neu_courses.parquet')
df = courses_df.copy()
df["prereq_count"] = df["prerequisite"].apply(len)

dept_avg = (
    df.groupby(['department name']).agg(
        n_courses = ('course_crn', 'nunique'),
        avg_prereq = ('prereq_count', 'mean')
    )
    .sort_values('avg_prereq', ascending=False)
)
dept_avg["avg_prereq"] = dept_avg["avg_prereq"].round(3) 

print(dept_avg.head(5))
# print(df.head())


                  n_courses  avg_prereq
department name                        
English                 114       3.474
Creative Writing         18       3.333
Anthropology             36       3.222
English Writing          19       3.158
Psychology               97       2.495


In [15]:
from networkx.algorithms.dag import dag_longest_path

def build_course_graph(course_df):
    G = nx.DiGraph()
    for _, row in courses_df.iterrows():
        course = row['course_crn']
        G.add_node(course, 
                   title = row['course_title']
                  )
        for prereq in row['prerequisite']:
            if prereq:
                G.add_node(prereq)
                G.add_edge(prereq, course)

    return G

In [16]:
G = build_course_graph(courses_df)

# Check if it’s a DAG
if nx.is_directed_acyclic_graph(G):
    longest_path = dag_longest_path(G)
else:
    # --- Show a few cycles to help debug the catalog anomalies
    print("Cycles found. Showing up to 5 example cycles:")
    for i, cyc in enumerate(nx.simple_cycles(G)):
        print("  " + " → ".join(cyc + [cyc[0]]))
        if i >= 4:
            break

    # --- Collapse SCCs to a DAG and take the longest path there
    C = nx.condensation(G)  # nodes are components (SCCs), edges preserve reachability
    comp_path = dag_longest_path(C)  # list of component IDs

    # pick a representative course code for each component for display
    def rep_component_node(comp_node):
        members = C.nodes[comp_node]["members"]  # set/list of original nodes in this SCC
        return sorted(members)[0]  # deterministic representative

    path_courses = [rep_component_node(c) for c in comp_path]

longest_len = max(0, len(path_courses) - 1)
print(f"\nLongest chain length (edges): {longest_len}")
print("Chain (start → … → end):")
for c in path_courses:
    t = G.nodes[c].get("title")
    print(f"  {c}" + (f" — {t}" if t else ""))

Cycles found. Showing up to 5 example cycles:
  SPNS 2101 → SPNS 2101
  SPNS 2102 → SPNS 2102
  SPNS 3101 → SPNS 3101
  SPNS 3102 → SPNS 3102
  SPNS 1101 → SPNS 1101

Longest chain length (edges): 9
Chain (start → … → end):
  BIOL 1107 — Foundations of Biology
  BIOL 1113 — General Biology 2
  PHSC 2301 — Human Physiology 1
  PHSC 2303 — Human Physiology 2
  HSCI 1105 — Human Nutrition
  NRSG 2220 — Health Assessment and Fundamental Nursing Skills
  NRSG 3320 — Nursing Care of Adults 1
  NRSG 3420 — Nursing Care of Adults 2
  NRSG 4502 — Nursing Care of the Child
  NRSG 4995 — Comprehensive Nursing Practicum
